# Automotive RAG Support Assistant (Skoda Domain)

This notebook implements an end-to-end Retrieval-Augmented Generation (RAG) conversational assistant for the **Automotive** domain. It ingests three official Skoda maintenance documentation files, splits them into semantic chunks, generates OpenAI embeddings, and stores them in a local FAISS vector store.

Environment Setup

In [ ]:
# Install modular packages, FAISS, and langchain-classic for chain imports
!pip install -q langchain langchain-core langchain-community langchain-openai langchain-classic faiss-cpu

import os
from google.colab import userdata

# 1. Configure Vocareum API Base URL
os.environ["OPENAI_BASE_URL"] = "https://openai.vocareum.com/v1"
os.environ["OPENAI_API_BASE"] = "https://openai.vocareum.com/v1"

# 2. Automatically retrieve Vocareum OpenAI API Key from Colab Secrets
try:
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("Successfully retrieved OPENAI_API_KEY from Colab Secrets.")
except Exception as e:
    raise ValueError(
        "Could not find 'OPENAI_API_KEY' in Colab Secrets! "
        "Please add it in the Secrets tab (key icon on the left sidebar)."
    )

print("Environment setup, Vocareum API base, and key configured successfully.")

Successfully retrieved OPENAI_API_KEY from Colab Secrets.
Environment setup, Vocareum API base, and key configured successfully.


## Section 1: Document Ingestion and Indexing (Skoda Maintenance Documents)
Created three official Skoda service and maintenance documentation files in the local `data/` folder, load them, split them into semantic chunks, generate OpenAI embeddings using Vocareum's endpoint, and index them using FAISS.

In [ ]:
import os
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Create data directory for official Skoda maintenance documents
os.makedirs("data", exist_ok=True)

skoda_official_docs = {
    "data/skoda_routine_service_schedule.txt": """
    SKODA OFFICIAL VEHICLE SERVICE & INTERVALS MANUAL
    Source: Official Skoda Maintenance and Service Documentation Schedule.
    - Standard Service Intervals: Mandatory routine maintenance is required every 15,000 km or 12 months, whichever occurs first, to keep vehicle warranties valid.
    - Minor/Interim Service: Includes engine oil replacement, oil filter element change, fluid level top-ups, and a full electronic diagnostic scan.
    - Major Service Milestones: Conducted every 30,000 km or 2 years. Includes replacement of spark plugs, cabin pollen filter cleaning/replacement, air filter element change, and fuel filter inspection.
    - Toothed Belt (Timing Belt) Check: Visual inspection of the timing belt and tensioning pulleys is required starting at 60,000 km, with replacement guidelines for diesel variants between 180,000 and 210,000 km.
    """,

    "data/skoda_brake_system_guide.txt": """
    SKODA BRAKE SYSTEM MAINTENANCE & REPAIR GUIDE
    Source: Skoda Workshop Manual - Chassis and Braking Systems.
    - Brake Pad Inspection: Front and rear brake pads must be inspected at every scheduled service interval. The minimum permissible friction lining thickness limit is strictly 3.0 mm. If thickness is lower, axle sets must be replaced immediately.
    - Brake Discs: Rotors should be checked for scoring, groove depth, and thickness variation. Discs falling below the manufacturer minimum thickness threshold must be swapped out.
    - Brake Fluid Flushing: Brake fluid absorbs atmospheric moisture over time; therefore, the hydraulic brake system must be completely flushed and refilled with fresh DOT 4 LV fluid every 2 years regardless of mileage.
    - Electronic Parking Brake (EPB): Vehicles featuring electronic parking brakes require service mode activation via the official diagnostic tool before attempting rear brake pad changes.
    """,

    "data/skoda_fluids_and_electrical.txt": """
    SKODA FLUID SPECIFICATIONS & ELECTRICAL MAINTENANCE GUIDE
    Source: Skoda Technical Information & Fluid Standards Handbook.
    - Engine Coolant Specification: Cooling systems require manufacturer-approved silicate-free coolant (G12evo or G13 specification) mixed strictly 50:50 with distilled water. Mixing incompatible coolants can cause engine block corrosion.
    - Transmission Service: Automatic Direct-Shift Gearboxes (DSG 7-speed dry clutch) use sealed lifetime fluid, whereas 6-speed wet DSG gearboxes require specialized transmission oil and internal filter changes every 60,000 km.
    - 12V Battery Health: Check battery terminal connection tightness, corrosion buildup, and open-circuit voltage during every service check. A resting voltage dipping below 12.4V indicates deep sulfation requiring a slow recharge or battery replacement.
    """
}

# Write the three documents to the local folder
for file_path, content in skoda_official_docs.items():
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(content.strip())
    print(f"Successfully created official Skoda document: {file_path}")

# 2. Load all 3 documents from the local folder
documents = []
for file_path in skoda_official_docs.keys():
    loader = TextLoader(file_path, encoding="utf-8")
    documents.extend(loader.load())

print(f"\nTotal source documents loaded: {len(documents)}")

# 3. Split content into semantic chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=40)
chunks = text_splitter.split_documents(documents)
print(f"Total semantic chunks generated: {len(chunks)}")

# 4. Generate OpenAI embeddings via Vocareum endpoint and store in FAISS vector store
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    base_url=os.environ.get("OPENAI_BASE_URL")
)
vector_store = FAISS.from_documents(chunks, embeddings)

# Save the FAISS index locally
vector_store.save_local("faiss_index")
print("Ingestion complete! FAISS vector store index saved locally to 'faiss_index'.")

Successfully created official Skoda document: data/skoda_routine_service_schedule.txt
Successfully created official Skoda document: data/skoda_brake_system_guide.txt
Successfully created official Skoda document: data/skoda_fluids_and_electrical.txt

Total source documents loaded: 3
Total semantic chunks generated: 10
Ingestion complete! FAISS vector store index saved locally to 'faiss_index'.


## Section 2: Retrieval-Augmented Chat & Context Awareness
Configured the history-aware retriever, retrieval chain, and strict system instructions to guarantee grounded answers and mandatory refusal behavior when information is missing.

In [ ]:
from langchain_classic.chains import create_history_aware_retriever, create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Load vector store index using Vocareum embeddings endpoint
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    base_url=os.environ.get("OPENAI_BASE_URL")
)
vector_store = FAISS.load_local(
    "faiss_index", embeddings, allow_dangerous_deserialization=True
)

# Retrieve top-k relevant chunks per query
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 3})

# Initialize LLM routing through Vocareum API base
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    base_url=os.environ.get("OPENAI_BASE_URL")
)

# Contextualize query prompt for follow-up support using conversation history
contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)
contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])
history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_q_prompt
)

# Strict grounding and mandatory refusal prompt
system_prompt = (
    "You are an automotive service assistant for Skoda vehicle maintenance tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer or if the answer "
    "is not contained within the provided documents, use exactly this "
    "response: 'I don’t have enough information in the provided documents.'\n\n"
    "{context}"
)
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)
rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)

print("Automotive RAG conversational chain successfully configured.")

Automotive RAG conversational chain successfully configured.


## Section 3: Sample Execution and Testing
Test the Automotive RAG assistant across three scenarios:
1. Standard retrieval query (Skoda service intervals)
2. Follow-up question using conversation history (Brake pad thickness)
3. Out-of-domain query triggering strict refusal behavior

In [ ]:
chat_history = []

test_queries = [
    "What are the standard service intervals for Skoda vehicles?",
    "What is the minimum permissible thickness limit for them?", # Follow-up test using context
    "What is the recipe for chocolate chip cookies?"              # Out-of-domain refusal test
]

for query in test_queries:
    print(f"User: {query}")
    response = rag_chain.invoke({"input": query, "chat_history": chat_history})
    print(f"Bot: {response['answer']}\n")

    # Update conversation history
    chat_history.append(("human", query))
    chat_history.append(("ai", response["answer"]))

User: What are the standard service intervals for Skoda vehicles?
Bot: Mandatory routine maintenance is required every 15,000 km or 12 months, whichever occurs first, to keep vehicle warranties valid.

User: What is the minimum permissible thickness limit for them?
Bot: The minimum permissible friction lining thickness limit for brake pads is strictly 3.0 mm. If the thickness is lower, axle sets must be replaced immediately.

User: What is the recipe for chocolate chip cookies?
Bot: I don’t have enough information in the provided documents.

